# Poem-GPT
## §2 — Corpus construction

In [ ]:
import re
import urllib.request
import csv
import io
import os
import json
import hashlib
from pathlib import Path

try:
    from unidecode import unidecode
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "Unidecode"], check=True)
    from unidecode import unidecode

PROJECT_DIR = Path(".")
DATA_DIR = PROJECT_DIR / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
CORPUS_PATH = DATA_DIR / "poems_corpus.txt"

SEED_BOOKS = {
    "whitman_leaves_of_grass": 1322, "dickinson_poems": 12242,
    "blake_songs": 1934, "blake_poems": 574, "poe_poems": 10031,
    "shakespeare_sonnets": 1041, "longfellow_hiawatha": 19,
    "fitzgerald_rubaiyat": 246, "housman_shropshire_lad": 5720,
    "coleridge_ancient_mariner": 151, "eliot_poems": 1567,
    "shelley_complete_poems": 4800, "tennyson_idylls": 610,
    "tennyson_early_poems": 8601, "rossetti_poems": 19188,
    "browning_sonnets_portuguese": 2002, "bryant_poems": 16341,
    "wordsworth_poetical_works_v2": 12145, "milton_paradise_lost": 20,
    "crabbe_the_village": 5203, "anthology_six_centuries": 30235,
}

CATALOG_URL = "https://www.gutenberg.org/cache/epub/feeds/pg_catalog.csv"
TARGET_CHARS = 40_000_000
MIN_BOOK_CHARS = 20_000

def discover_poetry_ids(exclude, limit=250):
    req = urllib.request.Request(CATALOG_URL, headers={"User-Agent": "poem-gpt-corpus-builder/1.0"})
    with urllib.request.urlopen(req, timeout=120) as resp:
        raw = resp.read().decode("utf-8", errors="replace")
    skip = re.compile(r"history|essay|criticism|letters|life of|memoir|dictionar|"
                      r"index|bibliograph|lecture|sermon|grammar|complete works|"
                      r"biograph|review|journal|correspondence", re.IGNORECASE)
    found = []
    for r in csv.DictReader(io.StringIO(raw)):
        if r["Type"] != "Text" or r["Language"] != "en":
            continue
        if "Poetry" not in r["Subjects"] or skip.search(r["Title"]):
            continue
        gid = int(r["Text#"])
        if gid not in exclude:
            found.append((gid, " ".join(r["Title"].split())[:60]))
    return found[:limit]

def strip_gutenberg_boilerplate(text: str) -> str:
    start_pat = re.compile(r"\*\*\*\s*START OF (THE|THIS) PROJECT GUTENBERG.*?\*\*\*", re.IGNORECASE | re.DOTALL)
    end_pat = re.compile(r"\*\*\*\s*END OF (THE|THIS) PROJECT GUTENBERG.*", re.IGNORECASE | re.DOTALL)
    m = start_pat.search(text)
    if m: text = text[m.end():]
    m = end_pat.search(text)
    if m: text = text[:m.start()]
    return text.strip()

def clean_poem_text(text: str) -> str:
    text = unidecode(text)
    text = text.replace("_", "")
    text = re.sub(r"\[\d+\]", "", text)
    kept = []
    for line in text.split("\n"):
        line = re.sub(r"\s{2,}\d+\s*$", "", line)
        s = line.strip()
        if re.match(r"^(PAGE\b|\[?Footnote|Cf\.|NOTES?\b|CONTENTS|INDEX|GLOSSARY|"
                    r"INTRODUCTION|PREFACE|APPENDIX|ILLUSTRATION)", s, re.IGNORECASE):
            continue
        if re.match(r"^[IVXLCDM]{2,}\.?$", s) or re.match(r"^[IVXLCDM]\.$", s):
            continue
        if s and not re.search(r"[a-z]", s) and re.search(r"[A-Z]{3}", s):
            continue
        if re.match(r"^[A-Z][A-Z '\-]{1,30}:$", s) or s.startswith("["):
            continue
        if re.match(r"^First (published|printed)\b", s, re.IGNORECASE):
            continue
        if re.match(r"^(l+\.|p+\.|\d+\.)\s", s):
            continue
        if s and sum(c in ".0123456789" for c in s) / len(s) > 0.5:
            continue
        kept.append(line)
    text = re.sub(r"\.{3,}", "", "\n".join(kept))
    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def fetch_book(gutenberg_id: int) -> str:
    url = f"https://www.gutenberg.org/cache/epub/{gutenberg_id}/pg{gutenberg_id}.txt"
    req = urllib.request.Request(url, headers={"User-Agent": "poem-gpt-corpus-builder/1.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        raw = resp.read().decode("utf-8", errors="replace")
    return clean_poem_text(strip_gutenberg_boilerplate(raw))

if CORPUS_PATH.exists():
    print(f"corpus already exists ({CORPUS_PATH.stat().st_size:,} bytes) -- skipping")
    text = CORPUS_PATH.read_text(encoding="utf-8")
else:
    all_text, total, used = [], 0, []
    for name, gid in SEED_BOOKS.items():
        try:
            t = fetch_book(gid); all_text.append(t); total += len(t); used.append((gid, name))
        except Exception as e:
            print(f"[seed] {name} FAILED: {e}")
    for gid, title in discover_poetry_ids(exclude=set(SEED_BOOKS.values())):
        if total >= TARGET_CHARS: break
        try: t = fetch_book(gid)
        except Exception: continue
        if len(t) < MIN_BOOK_CHARS: continue
        all_text.append(t); total += len(t); used.append((gid, title))
    text = "\n\n".join(all_text)
    CORPUS_PATH.write_text(text, encoding="utf-8")


## §3 — BPE tokenizer

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

TOKENIZER_PATH = DATA_DIR / "bpe_tokenizer.json"
TOKENIZER_META_PATH = DATA_DIR / "bpe_meta.json"

class BPETokenizer:
    def __init__(self, tokenizer): self._tok = tokenizer; self.vocab_size = tokenizer.get_vocab_size()

    @classmethod
    def train(cls, text, vocab_size=8000):
        tokenizer = Tokenizer(models.BPE())
        tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
        tokenizer.decoder = decoders.ByteLevel()
        trainer = trainers.BpeTrainer(vocab_size=vocab_size,
            initial_alphabet=pre_tokenizers.ByteLevel.alphabet(), show_progress=True)
        tokenizer.train_from_iterator([text], trainer=trainer)
        return cls(tokenizer)

    def encode(self, s): return self._tok.encode(s).ids
    def decode(self, ids): return self._tok.decode(ids)
    def save(self, path): self._tok.save(str(path))
    @classmethod
    def load(cls, path): return cls(Tokenizer.from_file(str(path)))

# build/load with corpus-hash cache invalidation
VOCAB_SIZE = 8000
CORPUS_HASH = hashlib.sha256(text.encode("utf-8")).hexdigest()[:16]
tok = None
if os.path.exists(TOKENIZER_PATH):
    cached_hash = json.load(open(TOKENIZER_META_PATH)).get("corpus_hash") if os.path.exists(TOKENIZER_META_PATH) else None
    tok = BPETokenizer.load(TOKENIZER_PATH)
    if cached_hash != CORPUS_HASH or tok.vocab_size > VOCAB_SIZE:
        tok = None
if tok is None:
    tok = BPETokenizer.train(text, vocab_size=VOCAB_SIZE)
    tok.save(TOKENIZER_PATH)
    json.dump({"corpus_hash": CORPUS_HASH, "vocab_size": tok.vocab_size}, open(TOKENIZER_META_PATH, "w"))


## §4 — Model

In [ ]:
import math, torch, torch.nn as nn, torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        assert n_embd % n_head == 0
        self.n_head = n_head; self.head_dim = n_embd // n_head
        self.qkv_proj = nn.Linear(n_embd, 3 * n_embd)
        self.out_proj = nn.Linear(n_embd, n_embd)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(block_size, block_size))
        self.register_buffer("causal_mask", mask.view(1, 1, block_size, block_size))

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.qkv_proj(x)
        q, k, v = qkv.split(C, dim=2)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        att = att.masked_fill(self.causal_mask[:, :, :T, :T] == 0, float("-inf"))
        att = self.attn_dropout(F.softmax(att, dim=-1))
        out = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))

class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(n_embd, 4*n_embd), nn.GELU(),
                                  nn.Linear(4*n_embd, n_embd), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, block_size, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd); self.attn = CausalSelfAttention(n_embd, n_head, block_size, dropout)
        self.ln2 = nn.LayerNorm(n_embd); self.mlp = MLP(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        return x + self.mlp(self.ln2(x))

class PoemGPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=256, n_head=4, n_layer=4, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.token_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)
        self.dropout = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([Block(n_embd, n_head, block_size, dropout) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.apply(self._init_weights)
        self.lm_head.weight = self.token_emb.weight  # weight tying

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.dropout(self.token_emb(idx) + self.pos_emb(pos))
        for block in self.blocks: x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1)) if targets is not None else None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, 1)], dim=1)
        return idx


## §6 — Training config (cosine LR + warmup)

In [ ]:
BLOCK_SIZE = 256; BATCH_SIZE = 64
N_EMBD = 256; N_HEAD = 4; N_LAYER = 4
LEARNING_RATE = 3e-4; WEIGHT_DECAY = 0.1
MAX_ITERS = 30000; EVAL_INTERVAL = 250; EVAL_ITERS = 50; SAMPLE_EVERY = 500; PATIENCE = 6
WARMUP_ITERS = 200; MIN_LR = 3e-5

def lr_at(it):
    if it < WARMUP_ITERS:
        return LEARNING_RATE * (it + 1) / WARMUP_ITERS
    progress = min(1.0, (it - WARMUP_ITERS) / max(1, MAX_ITERS - WARMUP_ITERS))
    return MIN_LR + 0.5 * (LEARNING_RATE - MIN_LR) * (1 + math.cos(math.pi * progress))

DROPOUT_LEVELS = [0.0, 0.1, 0.2]
MODEL_LABELS = {0.0: "Model A (dropout=0.0)", 0.1: "Model B (dropout=0.1)", 0.2: "Model C (dropout=0.2)"}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"

def checkpoint_path(dropout, best=False):
    tag = f"dropout_{dropout:.1f}" + ("_best.pt" if best else "_rolling.pt")
    return CHECKPOINT_DIR / f"model_{tag}"

def build_model(dropout):
    return PoemGPT(tok.vocab_size, BLOCK_SIZE, n_embd=N_EMBD, n_head=N_HEAD, n_layer=N_LAYER, dropout=dropout).to(device)


## §7 — Split (shuffled + pinned) and batching

In [ ]:
POEM_PROMPTS = {
    "poem_1": "The past is such a curious creature,\nTo look her in the face\nA transport may reward us,\nOr a",
    "poem_2": "I celebrate myself, and sing myself,\nAnd what I assume you shall assume,\nFor every atom belonging to me as",
    "poem_3": "Because I could not stop for Death,\nHe kindly stopped for me;\nThe carriage held but just ourselves",
}

SPLIT_CHUNK_TOKENS = 2048; SPLIT_SEED = 1337
all_tokens = torch.tensor(tok.encode(text), dtype=torch.long)
n_chunks = len(all_tokens) // SPLIT_CHUNK_TOKENS
chunks = all_tokens[:n_chunks * SPLIT_CHUNK_TOKENS].view(n_chunks, SPLIT_CHUNK_TOKENS)

def chunks_holding(prompt):
    pattern = r"\s+".join(re.escape(p) for p in re.split(r"\s+", prompt.strip()))
    m = re.search(pattern, text, re.IGNORECASE)
    c = len(tok.encode(text[:m.end()])) // SPLIT_CHUNK_TOKENS
    return {c, min(c + 1, n_chunks - 1)}

PINNED_CHUNKS = sorted(set().union(*(chunks_holding(p) for p in POEM_PROMPTS.values())))
perm = torch.randperm(n_chunks, generator=torch.Generator().manual_seed(SPLIT_SEED)).tolist()
n_val = n_chunks - int(0.9 * n_chunks)
rest = [c for c in perm if c not in set(PINNED_CHUNKS)]
val_idx, train_idx = rest[:n_val], PINNED_CHUNKS + rest[n_val:]

train_data = chunks[torch.tensor(train_idx)].reshape(-1).to(device)
val_data = chunks[torch.tensor(val_idx)].reshape(-1).to(device)
SPLIT_HASH = hashlib.sha256(str(sorted(val_idx)).encode()).hexdigest()[:8]
DATA_HASH = f"{CORPUS_HASH}:{SPLIT_HASH}"

_offsets = torch.arange(BLOCK_SIZE, device=device)
def get_batch(data, block_size, batch_size, device):
    ix = torch.randint(len(data) - block_size - 1, (batch_size,), device=data.device)
    window = ix[:, None] + _offsets[:block_size]
    return data[window], data[window + 1]


## §8 — Training loop

In [ ]:
@torch.no_grad()
def estimate_loss(model):
    model.eval()
    out = {}
    for split, d in [("train", train_data), ("val", val_data)]:
        losses = torch.zeros(EVAL_ITERS)
        for i in range(EVAL_ITERS):
            x, y = get_batch(d, BLOCK_SIZE, BATCH_SIZE, device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                _, loss = model(x, y)
            losses[i] = loss.item()
        out[split] = losses.mean().item()
    model.train()
    return out

def save_checkpoint(model, optimizer, step, dropout, path, val_loss=None, scaler=None):
    torch.save({"step": step, "val_loss": val_loss, "data_hash": DATA_HASH,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scaler_state_dict": scaler.state_dict() if scaler else None,
                "config": {"vocab_size": tok.vocab_size, "block_size": BLOCK_SIZE,
                           "n_embd": N_EMBD, "n_head": N_HEAD, "n_layer": N_LAYER, "dropout": dropout}}, path)

def train_model(dropout):
    label = MODEL_LABELS[dropout]
    model = build_model(dropout)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)
    rolling_path = checkpoint_path(dropout, best=False)
    best_path = checkpoint_path(dropout, best=True)
    start_step, best_val, no_improve = 0, float("inf"), 0

    if os.path.exists(rolling_path):
        ckpt = torch.load(rolling_path, map_location=device)
        if ckpt.get("data_hash") == DATA_HASH:
            model.load_state_dict(ckpt["model_state_dict"])
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
            start_step = ckpt["step"] + 1
            if os.path.exists(best_path):
                prev = torch.load(best_path, map_location=device)
                if prev.get("data_hash") == DATA_HASH:
                    best_val = prev["val_loss"]

    it = start_step
    for it in range(start_step, MAX_ITERS):
        for g in optimizer.param_groups: g["lr"] = lr_at(it)
        if it % EVAL_INTERVAL == 0 or it == MAX_ITERS - 1:
            losses = estimate_loss(model)
            if losses["val"] < best_val:
                best_val = losses["val"]; no_improve = 0
                save_checkpoint(model, optimizer, it, dropout, best_path, losses["val"], scaler)
            else:
                no_improve += 1
            save_checkpoint(model, optimizer, it, dropout, rolling_path, scaler=scaler)
            if no_improve >= PATIENCE: break
        x, y = get_batch(train_data, BLOCK_SIZE, BATCH_SIZE, device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    best = torch.load(best_path, map_location=device)
    model.load_state_dict(best["model_state_dict"])
    model.eval()
    return model, best_path, best["val_loss"]

trained_models = {}
for d in DROPOUT_LEVELS:
    trained_models[d], _, _ = train_model(d)


## §9 — Stimulus generation (seeded, trimmed, rejection-resampled)

In [ ]:
GEN_SEED = 20260829
torch.manual_seed(GEN_SEED)
GEN_TEMPERATURES = [0.6, 0.9, 1.2]
GEN_MAX_NEW_TOKENS = 60
GEN_TOP_K = None

def trim_to_last_line(s):
    cut = s.rstrip().rfind("\n")
    return s[:cut].rstrip() if cut > 0 else s.rstrip()

@torch.no_grad()
def generate_from(model, prompt, temperature, max_new_tokens=GEN_MAX_NEW_TOKENS, top_k=GEN_TOP_K):
    model.eval()
    ids = tok.encode(prompt)
    idx = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(idx, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)
    return tok.decode(out[0].tolist())

APPARATUS = re.compile(
    r"^\s*\d+\s+\S+\]"
    r"|\b(?:editions?|MS\.|MSS\.)\s+1[6-9]\d\d"
    r"|\s{2,}\*\w+\s*$"
    r"|^[ \t]*-[- \t]*$"
    r"|^[ \t]*[VXLCDM][ \t]*$",
    re.MULTILINE)
MAX_REDRAWS = 8

results = []; redraws = 0
for poem_id, prompt in POEM_PROMPTS.items():
    for dropout in DROPOUT_LEVELS:
        model = trained_models[dropout]
        for temperature in GEN_TEMPERATURES:
            continuation = ""
            for attempt in range(MAX_REDRAWS):
                gen_text = generate_from(model, prompt, temperature)
                continuation = re.sub(r"\n{3,}", "\n\n", trim_to_last_line(gen_text[len(prompt):]))
                if not APPARATUS.search(continuation): break
                redraws += 1
            results.append({"poem_id": poem_id, "model_label": MODEL_LABELS[dropout],
                             "dropout": dropout, "temperature": temperature, "prompt": prompt,
                             "continuation": continuation, "full_text": prompt + continuation})


## §9b — Novelty / surprise scoring

In [ ]:
NOVELTY_NGRAM_N = 4
POEM_CONTINUATION_END_MARKERS = {}

def ngrams(words, n): return [" ".join(words[i:i+n]) for i in range(len(words)-n+1)]

def ngram_overlap_pct(candidate_text, reference_ngram_set, n=NOVELTY_NGRAM_N):
    grams = ngrams(candidate_text.split(), n)
    return 100.0 * sum(g in reference_ngram_set for g in grams) / len(grams) if grams else 0.0

def find_span(prompt, corpus):
    pattern = r"\s+".join(re.escape(p) for p in re.split(r"\s+", prompt.strip()))
    return re.search(pattern, corpus, re.IGNORECASE)

def extract_true_continuation(poem_id, prompt, corpus, max_chars=800):
    m = find_span(prompt, corpus)
    if not m:
        return ""
    chunk = corpus[m.end():m.end() + max_chars]
    marker = POEM_CONTINUATION_END_MARKERS.get(poem_id)
    if marker and marker in chunk:
        return chunk[:chunk.index(marker) + len(marker)].strip()
    cut = re.search(r"\n\s*\n\s*\n", chunk)
    return (chunk[:cut.start()] if cut else chunk).strip()

POEM_TRUE_CONTINUATIONS = {pid: extract_true_continuation(pid, p, text) for pid, p in POEM_PROMPTS.items()}
POEM_TRUE_CONTINUATION_NGRAMS = {pid: set(ngrams(c.split(), 4)) for pid, c in POEM_TRUE_CONTINUATIONS.items() if c}
for pid, cont in POEM_TRUE_CONTINUATIONS.items():
    if cont:
        assert ngram_overlap_pct(cont, POEM_TRUE_CONTINUATION_NGRAMS[pid]) == 100.0

CORPUS_NGRAMS = set(ngrams(text.split(), 4))

@torch.no_grad()
def continuation_perplexity(model, prompt, continuation):
    prompt_ids = tok.encode(prompt); cont_ids = tok.encode(continuation)
    full_ids = (prompt_ids + cont_ids)[-(model.block_size + 1):]
    n_cont = min(len(cont_ids), len(full_ids) - 1)
    if n_cont <= 0:
        return float("nan")
    idx = torch.tensor([full_ids], dtype=torch.long, device=device)
    logits, _ = model(idx[:, :-1])
    targets = idx[:, 1:]
    logp = F.log_softmax(logits, dim=-1)
    token_logp = logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1)[0]
    return math.exp(-token_logp[-n_cont:].mean().item())

for row in results:
    ref_ngrams = POEM_TRUE_CONTINUATION_NGRAMS.get(row["poem_id"], set())
    row["ngram_overlap_original_pct"] = ngram_overlap_pct(row["continuation"], ref_ngrams)
    row["ngram_overlap_corpus_pct"] = ngram_overlap_pct(row["continuation"], CORPUS_NGRAMS)
    row["self_perplexity"] = continuation_perplexity(trained_models[row["dropout"]], row["prompt Ordinarily" if "prompt Ordinarily" in row else "prompt"], row["continuation"])
    cross = [continuation_perplexity(trained_models[d], row["prompt"], row["continuation"])
             for d in DROPOUT_LEVELS if d != row["dropout"]]
    row["cross_perplexity"] = sum(cross) / len(cross) if cross else float("nan")

print("Finished stimulus generation and scoring.")
